# Lab 10: Design and Optimize Prompts

> **Source:** Microsoft Learning -- [mslearn-genaiops](https://github.com/MicrosoftLearning/mslearn-genaiops), `docs/03-design-optimize-prompts.md`
> **License:** MIT

## Objectives

By the end of this lab you will:

1. Establish a **baseline** with standardized batch tests
2. Create **experiment branches** to isolate prompt changes
3. Apply a **manual scoring rubric** (1--5 scale)
4. Compare experiment results against baseline
5. Merge the winning experiment back to main

| Estimated Time | Estimated Cost |
|---|---|
| ~40 minutes | ~$1--2 |

## Experiment Workflow

![Architecture](architecture.png)

The key principle: **change one variable at a time**. Each experiment branch modifies a single aspect of the prompt (e.g., output format, token budget, persona) so you can attribute any quality or cost change to that specific modification.

## Step 1: Establish Baseline

Before optimizing, you need a **repeatable measurement**. We use 5 standardized test prompts that cover different trail-guide scenarios:

| Test ID | Prompt | Scenario |
|---|---|---|
| `day-hike-gear` | "What gear do I need for a day hike?" | Simple factual |
| `overnight-camping` | "I'm planning an overnight camping trip. What should I prepare?" | Planning |
| `three-day-backpacking` | "Help me plan a 3-day backpacking route." | Complex planning |
| `winter-hiking` | "What precautions should I take for winter hiking?" | Safety-critical |
| `trail-difficulty` | "How do I assess trail difficulty for beginners?" | Educational |

These same 5 prompts are used for **every experiment** -- the test set is the constant, the prompt is the variable.

In [ ]:
test_prompts = [
    {"id": "day-hike-gear", "prompt": "What gear do I need for a day hike?"},
    {"id": "overnight-camping", "prompt": "I'm planning an overnight camping trip. What should I prepare?"},
    {"id": "three-day-backpacking", "prompt": "Help me plan a 3-day backpacking route."},
    {"id": "winter-hiking", "prompt": "What precautions should I take for winter hiking?"},
    {"id": "trail-difficulty", "prompt": "How do I assess trail difficulty for beginners?"},
]
print(f"Test set: {len(test_prompts)} prompts")

## Step 2: Create Experiment Branch

Use the **branch-per-experiment** pattern. Each experiment gets its own Git branch so changes are isolated:

```bash
git checkout -b experiment/v4-concise
```

In this experiment, we modify the V3 prompt to produce **more concise responses** by:
- Replacing the 4-section framework with bullet-point format
- Lowering the token target from <500 to <300
- Adding "be direct, skip preamble" instruction

The branch name convention `experiment/<version>-<description>` makes it clear what is being tested.

## Step 3: Run Batch Tests

Run the same 5 test prompts against the modified agent. The script sends each prompt, collects the response, and records token usage:

```bash
python scripts/run_batch_tests.py <agent-id>
```

Or run inline from the notebook (next cell). The key metrics to capture per response are:
- **Response text** (for manual scoring)
- **Total tokens** (for cost comparison)

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

load_dotenv()
credential = DefaultAzureCredential()
project_client = AIProjectClient(
    credential=credential,
    endpoint=os.getenv("AZURE_AI_PROJECT_ENDPOINT"),
)

agent_id = os.getenv("AZURE_AI_AGENT_ID")
results = []
for test in test_prompts:
    thread = project_client.agents.create_thread()
    message = project_client.agents.create_message(
        thread_id=thread.id, role="user", content=test["prompt"]
    )
    run = project_client.agents.create_and_process_run(
        thread_id=thread.id, agent_id=agent_id
    )
    messages = project_client.agents.list_messages(thread_id=thread.id)
    response = messages.data[0].content[0].text.value
    results.append({"id": test["id"], "response": response, "tokens": run.usage.total_tokens})
    print(f"  {test['id']}: {run.usage.total_tokens} tokens")

print(f"\nAll {len(results)} tests complete")

## Step 4: Manual Scoring

Score each response on a **1--5 scale** across three dimensions:

| Dimension | 1 (Poor) | 3 (Adequate) | 5 (Excellent) |
|---|---|---|---|
| **Intent Resolution** | Ignores or misunderstands the question | Addresses the question but misses nuance | Fully resolves the user's intent with relevant detail |
| **Relevance** | Off-topic or generic filler | Mostly relevant but includes unnecessary content | Every sentence directly serves the user's need |
| **Groundedness** | Hallucinated facts or unsafe advice | Mix of accurate and unverifiable claims | All claims grounded in verifiable sources |

Record scores in CSV format for comparison:

```csv
test_id,version,intent_resolution,relevance,groundedness,tokens
day-hike-gear,v3,4,4,3,850
day-hike-gear,v4,4,5,4,490
overnight-camping,v3,4,4,4,920
overnight-camping,v4,4,4,4,510
...
```

> **Exam Tip:** Standardized test sets + a consistent scoring rubric = **controlled comparison**. This is the manual precursor to the automated evaluators in Lab 11.

## Step 5: Compare and Select Winner

Compare the V3 baseline against the V4 concise experiment:

| Metric | V3 (Baseline) | V4 (Concise) | Change |
|---|---|---|---|
| Avg Intent Resolution | 4.0 | 4.1 | +2.5% |
| Avg Relevance | 4.0 | 4.3 | +7.5% |
| Avg Groundedness | 3.6 | 3.8 | +5.6% |
| Avg Tokens | ~850 | ~490 | **-42%** |

**Result:** V4 achieves a **42% token reduction** (from ~850 to ~490 average tokens per response) while **maintaining or slightly improving** quality scores across all three dimensions.

> **Exam Tip:** Token reduction = **direct cost savings**. At GPT-4.1 pricing, a 42% token reduction across thousands of daily requests translates to significant operational savings. This is the business case for prompt optimization.

## Step 6: Merge Winner

The experiment succeeded -- merge V4 back to main and tag it:

```bash
git checkout main
git merge experiment/v4-concise
git tag -a v4 -m "V4: Concise formatting, 42% token reduction"
```

The experiment branch can be deleted after merging:

```bash
git branch -d experiment/v4-concise
```

This completes one cycle of the **experiment loop**. In production, you would run multiple experiments (e.g., `experiment/v4-persona`, `experiment/v4-few-shot`) and merge only the winners.

## Key Takeaways

| Concept | Detail |
|---|---|
| **Experiment Branches** | Isolate each prompt change in its own Git branch. Change one variable at a time. |
| **Standardized Test Sets** | The same 5 prompts are used for every experiment, ensuring apples-to-apples comparison. |
| **Manual Scoring Rubric** | 1--5 scale on Intent Resolution, Relevance, Groundedness. Consistent rubric = reproducible evaluation. |
| **Token Optimization** | Concise formatting reduced tokens by 42% with no quality loss -- a direct cost saving. |
| **Merge Winners Only** | Only winning experiments get merged and tagged. Failed experiments stay on their branches for reference. |

---

**Next:** [Lab 11 -- Automated Evaluation](../lab11-automated-evaluation/lab11-automated-evaluation.ipynb) -- replace manual scoring with cloud evaluators.